In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver07/data-project/sc_val.jsonl
/kaggle/input/datasets/achiver07/data-project/acts_val.jsonl
/kaggle/input/datasets/achiver07/data-project/acts_train.jsonl
/kaggle/input/datasets/achiver07/data-project/tribunals_val.jsonl
/kaggle/input/datasets/achiver07/data-project/test.json
/kaggle/input/datasets/achiver07/data-project/sc_train.jsonl
/kaggle/input/datasets/achiver07/data-project/tribunals_train.jsonl
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_val_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_train_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json
/kaggle/input/datasets/achiver07/phase1-token

In [2]:
# ================================================================
# ARCHITECTURE 4 — HYBRID GQA DECODER  (T4-optimised v4, multi-GPU)
# 201.44M params | 12 layers | d=1024 | 16Q/8KV | RoPE | SwiGLU
#
# RUN:
#   1 GPU :  python train_arch4_t4_v4.py
#   2 GPUs:  torchrun --standalone --nproc_per_node=2 train_arch4_t4_v4.py
#            (Kaggle "GPU T4 x2": use `!torchrun ...` in a notebook cell)
#
# v4 changes (on top of v3):
#  1. Data parallel across all GPUs: each GPU processes TOKENS_PER_STEP/world
#     tokens, gradients are averaged with one flat NCCL all-reduce per
#     optimizer step. The global batch stays 16,384 tokens/step, so the
#     training recipe (LR, steps, tokens) is unchanged -- just ~2x faster
#     on 2 GPUs.
#  2. Loss checkpointing is now OPTIONAL and chosen automatically: v3 always
#     recomputed the LM head in backward (~8% extra work). v4 tries the
#     faster no-recompute variants first and only checkpoints if it must.
#  3. Rank-safe batch choice, time limit, logging, checkpointing, resume.
#  Everything else (chunked local attention, compile trunk only, shared
#  RoPE/mask, fixed-window validation) is as in v3.
# ================================================================

import os, math, time, random, gc, queue, threading
from datetime import timedelta
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.utils.checkpoint import checkpoint

# ----------------------------------------------------------------
# Notebook re-run hygiene: if this script is re-run in the SAME kernel after
# a crash, the old model / optimizer / CUDA-graph pools / OOM traceback tensors
# are still on the GPU. Free what we can before doing anything else.
# (Restarting the session is still the cleanest fix.)
# ----------------------------------------------------------------
import sys
for _n in ("raw_model", "model", "compiled", "optimizer", "scaler", "loader",
           "trunk_fn", "total_loss", "x", "y", "loss", "logits", "lg", "ls"):
    globals().pop(_n, None)
sys.last_type = sys.last_value = sys.last_traceback = None
gc.collect()
try:
    torch._dynamo.reset()
    from torch._inductor import cudagraph_trees
    cudagraph_trees.reset_cudagraph_trees()
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ----------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------
CFG = {
    "DATA_DIR": "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1",
    "VOCAB_SIZE": 40000,

    "D_MODEL": 1024, "N_LAYERS": 12, "N_Q_HEADS": 16, "N_KV_HEADS": 8,
    "D_FF": 3328, "MAX_SEQ_LEN": 1024, "SEQ_LEN": 1024,
    "WINDOW": 256, "GLOBAL_EVERY": 3, "ROPE_BASE": 10000.0,
    "DROPOUT": 0.0, "INIT_STD": 0.02,

    "TOKENS_PER_STEP": 16384,          # GLOBAL tokens per optimizer step
    # (micro-batch, checkpoint_loss) tried in order, fastest first
    "CANDIDATES": [(8, False), (4, False), (8, True), (4, True), (2, True)],
    "MAX_STEPS": 10000,
    "LR": 3e-4, "MIN_LR_RATIO": 0.10, "WARMUP_STEPS": 500,
    "WEIGHT_DECAY": 0.1, "GRAD_CLIP": 1.0,

    "LOSS_CHUNK": 2048,
    "USE_FUSED_ADAMW": True,
    "USE_COMPILE": True,
    # "default" = fused kernels, NO CUDA graphs (robust, low memory).
    # "reduce-overhead" uses CUDA graphs: fragile with grad accumulation and
    # holds ~5 GB of extra private memory on a T4.
    "COMPILE_MODE": "default",

    "EVAL_EVERY": 500, "EVAL_BATCHES_PER_SOURCE": 10,
    "SAVE_EVERY": 500, "LOG_EVERY": 50,
    "TIME_LIMIT_HOURS": 11.0,
    "OUTPUT_DIR": "/kaggle/working/arch4_hybrid_gqa_201m",
    "RESUME_FROM": None,
    "SEED": 42,
}

SOURCE_NAMES = ["acts", "sc", "tribunals"]
SOURCE_PROBS = np.array([0.273, 0.469, 0.258], dtype=np.float64)
SOURCE_PROBS /= SOURCE_PROBS.sum()

# ----------------------------------------------------------------
# DISTRIBUTED SETUP
# ----------------------------------------------------------------
RANK = int(os.environ.get("RANK", 0))
WORLD = int(os.environ.get("WORLD_SIZE", 1))
LOCAL = int(os.environ.get("LOCAL_RANK", 0))
DIST = WORLD > 1
IS_MAIN = RANK == 0

if DIST:
    torch.cuda.set_device(LOCAL)
    dist.init_process_group("nccl", timeout=timedelta(minutes=60))
DEVICE = torch.device("cuda", LOCAL) if torch.cuda.is_available() else torch.device("cpu")


def log(*a):
    if IS_MAIN:
        print(*a, flush=True)


WALL_START = time.time()
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
p = torch.cuda.get_device_properties(DEVICE) if DEVICE.type == "cuda" else None
log(f"Device: {DEVICE} | world size {WORLD} | visible GPUs {n_gpus}")
if p is not None:
    log(f"GPU: {p.name} {p.total_memory / 1024**3:.2f} GB")
    _free, _total = torch.cuda.mem_get_info(DEVICE)
    log(f"GPU memory free at start: {_free / 1024**3:.2f} / {_total / 1024**3:.2f} GB")
    if _free < 0.9 * _total:
        log("*** WARNING: GPU memory is already in use (leftover from a previous run in "
            "this notebook kernel, or another process). Restart the session "
            "(Run > Restart session) and run again, otherwise every batch size will fail. ***")
if not DIST and n_gpus > 1:
    log(f"*** {n_gpus} GPUs detected but only 1 is used. For ~{n_gpus}x speed run:\n"
        f"    torchrun --standalone --nproc_per_node={n_gpus} <this_script>.py ***")

torch.set_float32_matmul_precision("high")
assert CFG["TOKENS_PER_STEP"] % WORLD == 0
TOKENS_PER_RANK = CFG["TOKENS_PER_STEP"] // WORLD


def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


seed_everything(CFG["SEED"])          # same seed on all ranks -> identical init
OUT = CFG["OUTPUT_DIR"]
if IS_MAIN:
    os.makedirs(OUT, exist_ok=True)

# ----------------------------------------------------------------
# DATA
# ----------------------------------------------------------------
D = CFG["DATA_DIR"]
PATHS = {
    "acts_train": f"{D}/acts_train.bin", "sc_train": f"{D}/sc_train.bin",
    "tribunals_train": f"{D}/tribunals_train_f.bin",
    "acts_val": f"{D}/acts_val.bin", "sc_val": f"{D}/sc_val.bin",
    "tribunals_val": f"{D}/tribunals_val_f.bin",
}
for k, v in PATHS.items():
    assert os.path.exists(v), f"Missing {k}: {v}"

train_arrays = {n: np.memmap(PATHS[f"{n}_train"], dtype=np.uint16, mode="r") for n in SOURCE_NAMES}
val_arrays = {n: np.memmap(PATHS[f"{n}_val"], dtype=np.uint16, mode="r") for n in SOURCE_NAMES}
for n in SOURCE_NAMES:
    log(f"{n:10s} train {len(train_arrays[n]):>12,} | val {len(val_arrays[n]):>12,}")


def fill_batch(buf, arrays, rng, seq_len):
    srcs = rng.choice(len(SOURCE_NAMES), size=buf.shape[0], p=SOURCE_PROBS)
    for b, s in enumerate(srcs):
        d = arrays[SOURCE_NAMES[s]]
        st = rng.randint(0, len(d) - seq_len)
        buf[b] = d[st:st + seq_len + 1]


def to_xy(t):
    t = t.to(DEVICE, non_blocking=True).long()
    return t[:, :-1].contiguous(), t[:, 1:].contiguous()


class Loader:
    """Background thread preparing pinned int32 batches (different data per rank)."""

    def __init__(self, arrays, batch_size, seq_len, seed, depth=8):
        self.arrays, self.bs, self.T = arrays, batch_size, seq_len
        self.rng = np.random.RandomState(seed)
        self.q = queue.Queue(depth)
        threading.Thread(target=self._work, daemon=True).start()

    def _work(self):
        if DEVICE.type == "cuda":
            torch.cuda.set_device(DEVICE)          # pin on THIS rank's GPU context
        while True:
            buf = np.empty((self.bs, self.T + 1), dtype=np.int32)
            fill_batch(buf, self.arrays, self.rng, self.T)
            t = torch.from_numpy(buf)
            self.q.put(t.pin_memory() if DEVICE.type == "cuda" else t)

    def next(self):
        return to_xy(self.q.get())


# ----------------------------------------------------------------
# MODEL
# ----------------------------------------------------------------
def rotate_half(x):
    h = x.shape[-1] // 2
    return torch.cat([-x[..., h:], x[..., :h]], dim=-1)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base):
        super().__init__()
        inv = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        f = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv)
        emb = torch.cat([f, f], dim=-1)
        self.register_buffer("cos", emb.cos(), persistent=False)
        self.register_buffer("sin", emb.sin(), persistent=False)

    def forward(self, T):
        return self.cos[:T][None, None], self.sin[:T][None, None]


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.in_proj = nn.Linear(d_model, 2 * d_ff, bias=False)
        self.out_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        g, u = self.in_proj(x).chunk(2, dim=-1)
        return self.out_proj(F.silu(g) * u)


class GQAAttention(nn.Module):
    def __init__(self, d_model, n_q, n_kv, window, is_global, dropout=0.0):
        super().__init__()
        assert d_model % n_q == 0 and n_q % n_kv == 0
        self.n_q, self.n_kv = n_q, n_kv
        self.head_dim = d_model // n_q
        self.rep = n_q // n_kv
        self.window, self.is_global, self.dropout = window, is_global, dropout
        q_dim, kv_dim = n_q * self.head_dim, n_kv * self.head_dim
        self.split = [q_dim, kv_dim, kv_dim]
        self.qkv_proj = nn.Linear(d_model, q_dim + 2 * kv_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin, mask):
        B, T, C = x.shape
        Hq, Hkv, Dh, rep = self.n_q, self.n_kv, self.head_dim, self.rep
        q, k, v = torch.split(self.qkv_proj(x), self.split, dim=-1)
        q = q.view(B, T, Hq, Dh).transpose(1, 2)
        k = k.view(B, T, Hkv, Dh).transpose(1, 2)
        v = v.view(B, T, Hkv, Dh).transpose(1, 2)

        cos, sin = cos.to(q.dtype), sin.to(q.dtype)
        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin
        dp = self.dropout if self.training else 0.0

        if self.is_global:
            k = k[:, :, None].expand(B, Hkv, rep, T, Dh).reshape(B, Hq, T, Dh)
            v = v[:, :, None].expand(B, Hkv, rep, T, Dh).reshape(B, Hq, T, Dh)
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dp, is_causal=True)
            return self.out_proj(y.transpose(1, 2).reshape(B, T, C))

        W = self.window
        N = T // W
        qc = q.view(B, Hq, N, W, Dh).transpose(1, 2).reshape(B * N, Hq, W, Dh)

        def chunk_kv(t):
            tc = t.view(B, Hkv, N, W, Dh)
            prev = F.pad(tc[:, :, :-1], (0, 0, 0, 0, 1, 0))
            t2 = torch.cat([prev, tc], dim=3)
            t2 = t2.transpose(1, 2)[:, :, :, None]
            return t2.expand(B, N, Hkv, rep, 2 * W, Dh).reshape(B * N, Hq, 2 * W, Dh)

        y = F.scaled_dot_product_attention(qc, chunk_kv(k), chunk_kv(v),
                                           attn_mask=mask, dropout_p=dp)
        y = y.view(B, N, Hq, W, Dh).permute(0, 1, 3, 2, 4).reshape(B, T, C)
        return self.out_proj(y)


class Arch4Block(nn.Module):
    def __init__(self, cfg, idx):
        super().__init__()
        self.is_global = (idx + 1) % cfg["GLOBAL_EVERY"] == 0
        self.norm1 = nn.LayerNorm(cfg["D_MODEL"])
        self.attn = GQAAttention(cfg["D_MODEL"], cfg["N_Q_HEADS"], cfg["N_KV_HEADS"],
                                 cfg["WINDOW"], self.is_global, cfg["DROPOUT"])
        self.norm2 = nn.LayerNorm(cfg["D_MODEL"])
        self.ffn = SwiGLU(cfg["D_MODEL"], cfg["D_FF"])

    def forward(self, x, cos, sin, mask):
        x = x + self.attn(self.norm1(x), cos, sin, mask)
        return x + self.ffn(self.norm2(x))


class Architecture4(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        W = cfg["WINDOW"]
        self.window = W
        assert cfg["MAX_SEQ_LEN"] % W == 0 and cfg["SEQ_LEN"] % W == 0

        self.token_embedding = nn.Embedding(cfg["VOCAB_SIZE"], cfg["D_MODEL"])
        self.layers = nn.ModuleList([Arch4Block(cfg, i) for i in range(cfg["N_LAYERS"])])
        self.final_norm = nn.LayerNorm(cfg["D_MODEL"])
        self.lm_head = nn.Linear(cfg["D_MODEL"], cfg["VOCAB_SIZE"], bias=False)
        self.lm_head.weight = self.token_embedding.weight

        self.rope = RotaryEmbedding(cfg["D_MODEL"] // cfg["N_Q_HEADS"],
                                    cfg["MAX_SEQ_LEN"], cfg["ROPE_BASE"])
        n_chunks = cfg["MAX_SEQ_LEN"] // W
        i = torch.arange(W)[:, None]
        j = torch.arange(2 * W)[None, :]
        m = (j > i) & (j <= i + W)
        mask = m[None].repeat(n_chunks, 1, 1)
        mask[0] &= (j >= W)
        self.register_buffer("chunk_mask", mask[:, None], persistent=False)

        self._init_weights()

    def _init_weights(self):
        std = self.cfg["INIT_STD"]
        nn.init.normal_(self.token_embedding.weight, 0.0, std)
        for m in self.modules():
            if isinstance(m, nn.Linear) and m.weight is not self.token_embedding.weight:
                nn.init.normal_(m.weight, 0.0, std)
        s = 1.0 / math.sqrt(2 * self.cfg["N_LAYERS"])
        for b in self.layers:
            b.attn.out_proj.weight.data.mul_(s)
            b.ffn.out_proj.weight.data.mul_(s)

    def hidden_states(self, input_ids):
        B, T = input_ids.shape
        W = self.window
        assert T % W == 0
        N = T // W
        x = self.token_embedding(input_ids)
        cos, sin = self.rope(T)
        mask = self.chunk_mask[:N][None].expand(B, N, 1, W, 2 * W).reshape(B * N, 1, W, 2 * W)
        for b in self.layers:
            x = b(x, cos, sin, mask)
        return self.final_norm(x)

    def forward(self, input_ids):
        return self.lm_head(self.hidden_states(input_ids))


class Trunk(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, ids):
        return self.m.hidden_states(ids)


def chunked_ce(hidden, targets, weight, chunk, ckpt):
    """Mean CE. ckpt=True: never stores full logits (recomputes in backward).
    ckpt=False: plain single-pass loss (fastest, uses more memory).
    Under no_grad (eval) it simply loops over chunks to keep memory low."""
    h = hidden.reshape(-1, hidden.size(-1))
    t = targets.reshape(-1)
    N = h.size(0)

    def f(hc, tc, w):
        return F.cross_entropy(F.linear(hc, w).float(), tc, reduction="sum")

    grad = torch.is_grad_enabled()
    if grad and not ckpt:
        return f(h, t, weight) / N
    total = h.new_zeros((), dtype=torch.float32)
    for i in range(0, N, chunk):
        args = (h[i:i + chunk], t[i:i + chunk], weight)
        total = total + (checkpoint(f, *args, use_reentrant=False) if grad else f(*args))
    return total / N


log("\nBuilding model...")
raw_model = Architecture4(CFG).to(DEVICE)
total_params = sum(p.numel() for p in raw_model.parameters())
log(f"Parameters: {total_params:,} ({total_params / 1e6:.3f}M)")
log("Matches spec (~201.44M):", total_params == 201_443_328)
log("Attention:", " ".join("G" if b.is_global else "L" for b in raw_model.layers))

# ----------------------------------------------------------------
# OPTIMIZER
# ----------------------------------------------------------------
decay, no_decay = [], []
for n, p in raw_model.named_parameters():
    (decay if p.ndim >= 2 and "norm" not in n.lower() and "embedding" not in n.lower()
     else no_decay).append(p)
groups = [{"params": decay, "weight_decay": CFG["WEIGHT_DECAY"]},
          {"params": no_decay, "weight_decay": 0.0}]
opt_kw = dict(lr=CFG["LR"], betas=(0.9, 0.95), eps=1e-8)
use_fused = False
if DEVICE.type == "cuda" and CFG["USE_FUSED_ADAMW"]:
    try:
        optimizer = torch.optim.AdamW(groups, fused=True, **opt_kw)
        use_fused = True
    except Exception as e:
        log("Fused AdamW unavailable:", str(e)[:200])
        optimizer = torch.optim.AdamW(groups, foreach=True, **opt_kw)
else:
    optimizer = torch.optim.AdamW(groups, foreach=True, **opt_kw)
log("Fused AdamW:", use_fused)

PARAMS = list(raw_model.parameters())
USE_AMP = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def autocast():
    return torch.amp.autocast("cuda", enabled=USE_AMP, dtype=torch.float16)


def get_lr(step):
    if step < CFG["WARMUP_STEPS"]:
        return CFG["LR"] * (step + 1) / CFG["WARMUP_STEPS"]
    prog = (step - CFG["WARMUP_STEPS"]) / (CFG["MAX_STEPS"] - CFG["WARMUP_STEPS"])
    prog = min(max(prog, 0.0), 1.0)
    lo = CFG["LR"] * CFG["MIN_LR_RATIO"]
    return lo + (CFG["LR"] - lo) * 0.5 * (1 + math.cos(math.pi * prog))


def memory_report(tag=""):
    if DEVICE.type != "cuda":
        return
    log(f"GPU mem {tag}| alloc={torch.cuda.memory_allocated(DEVICE) / 1024**3:.2f} GB | "
        f"reserved={torch.cuda.memory_reserved(DEVICE) / 1024**3:.2f} GB | "
        f"peak={torch.cuda.max_memory_allocated(DEVICE) / 1024**3:.2f} GB")


def allreduce_grads():
    """Average gradients over ranks with ONE flat all-reduce.
    If any rank has inf/nan the sum propagates it, so every rank's GradScaler
    skips the same step and weights stay in sync."""
    if not DIST:
        return
    for p in PARAMS:
        if p.grad is None:
            p.grad = torch.zeros_like(p)
    grads = [p.grad for p in PARAMS]
    flat = torch._utils._flatten_dense_tensors(grads)
    dist.all_reduce(flat)
    flat.div_(WORLD)
    for g, s in zip(grads, torch._utils._unflatten_dense_tensors(flat, grads)):
        g.copy_(s)
    del flat


# ----------------------------------------------------------------
# INIT SANITY CHECK
# ----------------------------------------------------------------
raw_model.eval()
b0 = np.empty((1, CFG["SEQ_LEN"] + 1), dtype=np.int32)
fill_batch(b0, train_arrays, np.random.RandomState(0), CFG["SEQ_LEN"])
x, y = to_xy(torch.from_numpy(b0))
with torch.no_grad(), autocast():
    lg = raw_model(x)
    ls = F.cross_entropy(lg.float().reshape(-1, lg.size(-1)), y.reshape(-1))
log(f"\nInit loss {ls.item():.4f} (random CE {math.log(CFG['VOCAB_SIZE']):.4f}) | "
    f"logits std {lg.float().std().item():.3f}")
del lg, ls, x, y
gc.collect(); torch.cuda.empty_cache()

# ----------------------------------------------------------------
# FORWARD + LOSS
# ----------------------------------------------------------------
trunk_fn = raw_model.hidden_states
compile_enabled = False
LOSS_CKPT = False


def mark_iteration():
    if (compile_enabled and CFG["COMPILE_MODE"] == "reduce-overhead"
            and hasattr(torch.compiler, "cudagraph_mark_step_begin")):
        torch.compiler.cudagraph_mark_step_begin()


def fwd_loss(x, y):
    h = trunk_fn(x)
    return chunked_ce(h, y, raw_model.lm_head.weight, CFG["LOSS_CHUNK"], LOSS_CKPT)


# ----------------------------------------------------------------
# AUTO CONFIG: fastest (micro-batch, loss-ckpt) that fits on EVERY rank
# ----------------------------------------------------------------
def try_config(bs, ckpt):
    global LOSS_CKPT
    LOSS_CKPT = ckpt
    ok, peak = True, None
    try:
        raw_model.train()
        optimizer.zero_grad(set_to_none=True)
        torch.cuda.reset_peak_memory_stats(DEVICE)
        b = np.empty((bs, CFG["SEQ_LEN"] + 1), dtype=np.int32)
        fill_batch(b, train_arrays, np.random.RandomState(1), CFG["SEQ_LEN"])
        x, y = to_xy(torch.from_numpy(b))
        with autocast():
            loss = fwd_loss(x, y)
        scaler.scale(loss).backward()
        torch.cuda.synchronize()
        peak = torch.cuda.max_memory_allocated(DEVICE) / 1024**3
        # AdamW states (8 B/param) + flat all-reduce buffer (4 B/param) are not
        # allocated yet during this test, so reserve headroom for them.
        total_gb = torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3
        if peak + 12 * total_params / 1024**3 > 0.97 * total_gb:
            ok = False
    except torch.cuda.OutOfMemoryError:
        ok = False
    optimizer.zero_grad(set_to_none=True)
    gc.collect(); torch.cuda.empty_cache()
    return ok, peak


def select_config():
    """Fastest (micro-batch, loss-ckpt) that fits on EVERY rank (eager test)."""
    cands = [(bs, ck) for bs, ck in CFG["CANDIDATES"]
             if TOKENS_PER_RANK % (bs * CFG["SEQ_LEN"]) == 0]
    log(f"\nTesting configs (per-GPU tokens/step = {TOKENS_PER_RANK:,})...")
    chosen = len(cands)
    for idx, (bs, ck) in enumerate(cands):
        ok, peak = try_config(bs, ck)
        log(f"  batch {bs} ckpt_loss={ck}: {'OK, peak %.2f GB' % peak if ok else 'too big / OOM'}")
        if ok:
            chosen = idx
            break
    if DIST:                               # most conservative choice across ranks
        t = torch.tensor([chosen], device=DEVICE)
        dist.all_reduce(t, op=dist.ReduceOp.MAX)
        chosen = int(t.item())
    if chosen >= len(cands):
        _free, _total = torch.cuda.mem_get_info(DEVICE)
        raise RuntimeError(
            f"No candidate config fits. GPU free now: {_free / 1024**3:.2f} / "
            f"{_total / 1024**3:.2f} GB. If this is much less than ~14 GB, memory is held by a "
            f"previous run in this kernel: RESTART the session and run again.")
    return cands[chosen]


def describe_config():
    log(f"Using per-GPU batch {BATCH} x accum {GRAD_ACCUM} x {WORLD} GPU(s) = "
        f"{tokens_per_step:,} tok/step | loss ckpt: {LOSS_CKPT} | "
        f"total {tokens_per_step * CFG['MAX_STEPS']:,} tokens")


BATCH, LOSS_CKPT = select_config()
GRAD_ACCUM = TOKENS_PER_RANK // (BATCH * CFG["SEQ_LEN"])
tokens_per_step = BATCH * CFG["SEQ_LEN"] * GRAD_ACCUM * WORLD   # global
describe_config()

# ----------------------------------------------------------------
# RESUME + rank sync
# ----------------------------------------------------------------
start_step, best_val = 0, float("inf")
if CFG["RESUME_FROM"] and os.path.exists(CFG["RESUME_FROM"]):
    ck = torch.load(CFG["RESUME_FROM"], map_location=DEVICE)
    raw_model.load_state_dict(ck["model"])
    if "optimizer" in ck:
        optimizer.load_state_dict(ck["optimizer"])
        scaler.load_state_dict(ck["scaler"])
    start_step, best_val = ck["step"], ck["best_val"]
    del ck
    gc.collect(); torch.cuda.empty_cache()
    log(f"Resumed from step {start_step}")

if DIST:                                   # guarantee identical weights on all ranks
    for p in PARAMS:
        dist.broadcast(p.data, 0)

loader = Loader(train_arrays, BATCH, CFG["SEQ_LEN"], CFG["SEED"] + start_step + 1000 * RANK)


def micro_steps():
    total = torch.zeros((), device=DEVICE)
    for _ in range(GRAD_ACCUM):
        x, y = loader.next()
        mark_iteration()
        with autocast():
            loss = fwd_loss(x, y)
        scaler.scale(loss / GRAD_ACCUM).backward()
        total += loss.detach()
    return total


# ----------------------------------------------------------------
# COMPILE (trunk only) + warm-up mirroring the real loop (no comms here)
# ----------------------------------------------------------------
compiled = None


def release_compile_state():
    """Drop compiled graphs and CUDA-graph private pools so the eager fallback
    gets the whole GPU back (a failed reduce-overhead attempt otherwise keeps
    several GB allocated)."""
    global trunk_fn, compile_enabled, compiled
    trunk_fn, compile_enabled, compiled = raw_model.hidden_states, False, None
    optimizer.zero_grad(set_to_none=True)
    try:
        torch._dynamo.reset()
    except Exception:
        pass
    try:
        from torch._inductor import cudagraph_trees
        cudagraph_trees.reset_cudagraph_trees()
    except Exception:
        pass
    gc.collect(); torch.cuda.empty_cache()


if CFG["USE_COMPILE"] and DEVICE.type == "cuda" and hasattr(torch, "compile"):
    log(f"\nCompiling trunk, mode={CFG['COMPILE_MODE']} (a few minutes; the "
        f"'max_autotune_gemm' SM warning is harmless)...")
    compile_ok = True
    try:
        compiled = torch.compile(Trunk(raw_model), mode=CFG["COMPILE_MODE"], dynamic=False)
        trunk_fn, compile_enabled = compiled, True
        raw_model.train()
        for _ in range(2):                  # 2 full accumulated steps (no comms here)
            optimizer.zero_grad(set_to_none=True)
            micro_steps()
        torch.cuda.synchronize()
        optimizer.zero_grad(set_to_none=True)
    except Exception as e:
        compile_ok = False
        print(f"[rank {RANK}] compile/warm-up failed. Reason: {str(e)[:600]}", flush=True)

    if DIST:                                # all ranks must take the same path
        t = torch.tensor([1 if compile_ok else 0], device=DEVICE)
        dist.all_reduce(t, op=dist.ReduceOp.MIN)
        compile_ok = bool(t.item())

    if compile_ok:
        log("torch.compile: ENABLED")
        gc.collect(); torch.cuda.empty_cache()
    else:
        log("torch.compile: FAILED -> eager fallback (freeing compiled state, re-checking batch size)")
        release_compile_state()
        old_batch = BATCH
        BATCH, LOSS_CKPT = select_config()
        GRAD_ACCUM = TOKENS_PER_RANK // (BATCH * CFG["SEQ_LEN"])
        tokens_per_step = BATCH * CFG["SEQ_LEN"] * GRAD_ACCUM * WORLD
        if BATCH != old_batch:
            loader = Loader(train_arrays, BATCH, CFG["SEQ_LEN"],
                            CFG["SEED"] + start_step + 1000 * RANK)
        describe_config()
else:
    log("\ntorch.compile: DISABLED")


# ----------------------------------------------------------------
# VALIDATION (rank 0 only, eager, fixed windows)
# ----------------------------------------------------------------
@torch.no_grad()
def evaluate():
    raw_model.eval()
    rng = np.random.RandomState(1234)
    T = CFG["SEQ_LEN"]
    res = {}
    for name in SOURCE_NAMES:
        data, losses = val_arrays[name], []
        for _ in range(CFG["EVAL_BATCHES_PER_SOURCE"]):
            b = np.empty((BATCH, T + 1), dtype=np.int32)
            for i in range(BATCH):
                st = rng.randint(0, len(data) - T)
                b[i] = data[st:st + T + 1]
            x, y = to_xy(torch.from_numpy(b))
            with autocast():
                h = raw_model.hidden_states(x)
                loss = chunked_ce(h, y, raw_model.lm_head.weight, CFG["LOSS_CHUNK"], False)
            losses.append(loss.item())
        res[name] = float(np.mean(losses))
    res["mean"] = float(sum(SOURCE_PROBS[i] * res[n] for i, n in enumerate(SOURCE_NAMES)))
    raw_model.train()
    return res


def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def save_latest(step):
    atomic_save({"step": step, "model": raw_model.state_dict(),
                 "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                 "best_val": best_val, "cfg": CFG}, os.path.join(OUT, "latest.pt"))
    log(f"Saved latest.pt @ step {step}")


# ----------------------------------------------------------------
# TRAIN
# ----------------------------------------------------------------
log("\n" + "=" * 70 + "\nSTARTING TRAINING\n" + "=" * 70)
raw_model.train()
optimizer.zero_grad(set_to_none=True)
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(DEVICE)

t0 = time.time()
steps_done = 0
last_step = start_step
time_limit = CFG["TIME_LIMIT_HOURS"] * 3600

for step in range(start_step + 1, CFG["MAX_STEPS"] + 1):
    optimizer.zero_grad(set_to_none=True)
    total_loss = micro_steps()

    allreduce_grads()                              # average grads across GPUs
    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), CFG["GRAD_CLIP"], foreach=True)
    lr = get_lr(step - 1)
    for g in optimizer.param_groups:
        g["lr"] = lr
    scaler.step(optimizer)
    scaler.update()
    steps_done += 1
    last_step = step

    if step == start_step + 1 or step % CFG["LOG_EVERY"] == 0:
        if DIST:
            dist.all_reduce(total_loss)
            total_loss /= WORLD
        el = time.time() - t0
        tps = steps_done * tokens_per_step / max(el, 1e-6)
        eta_h = (CFG["MAX_STEPS"] - step) * tokens_per_step / max(tps, 1) / 3600
        log(f"STEP {step:5d}/{CFG['MAX_STEPS']} | loss {total_loss.item() / GRAD_ACCUM:.4f} | "
            f"lr {lr:.2e} | grad {float(grad_norm):.2f} | {tps:,.0f} tok/s | ETA {eta_h:.2f} h")

    if IS_MAIN and step % CFG["EVAL_EVERY"] == 0:
        val = evaluate()
        print(f"\nSTEP {step} VAL | mean {val['mean']:.4f} | acts {val['acts']:.4f} | "
              f"sc {val['sc']:.4f} | tribunals {val['tribunals']:.4f}", flush=True)
        memory_report()
        if val["mean"] < best_val:
            best_val = val["mean"]
            atomic_save({"step": step, "model": raw_model.state_dict(),
                         "best_val": best_val, "cfg": CFG}, os.path.join(OUT, "best.pt"))
            print("NEW BEST ->", os.path.join(OUT, "best.pt"), "\n", flush=True)

    if IS_MAIN and step % CFG["SAVE_EVERY"] == 0:
        save_latest(step)

    if step % 10 == 0:                              # rank-consistent stop decision
        flag = torch.tensor([1.0 if (IS_MAIN and time.time() - WALL_START > time_limit) else 0.0],
                            device=DEVICE)
        if DIST:
            dist.broadcast(flag, 0)
        if flag.item() > 0:
            log(f"\nTime limit ({CFG['TIME_LIMIT_HOURS']} h) reached at step {step}. "
                f"Saving; set RESUME_FROM to continue.")
            break

if IS_MAIN:
    save_latest(last_step)
if DIST:
    dist.barrier()
el = time.time() - t0
log("\n" + "=" * 70 + "\nTRAINING STOPPED / COMPLETE\n" + "=" * 70)
log(f"Last step {last_step} | best val {best_val:.4f} | time {el / 3600:.2f} h | "
    f"{steps_done * tokens_per_step / max(el, 1e-6):,.0f} tok/s (all GPUs)")
memory_report()
if DIST:
    dist.destroy_process_group()

Device: cuda:0 | world size 1 | visible GPUs 2
GPU: Tesla T4 14.56 GB
GPU memory free at start: 14.46 / 14.56 GB
*** 2 GPUs detected but only 1 is used. For ~2x speed run:
    torchrun --standalone --nproc_per_node=2 <this_script>.py ***
acts       train  145,554,561 | val    3,368,151
sc         train  249,782,556 | val    4,799,399
tribunals  train  137,567,074 | val   28,404,707

Building model...
Parameters: 201,443,328 (201.443M)
Matches spec (~201.44M): True
Attention: L L G L L G L L G L L G
Fused AdamW: True

Init loss 10.8820 (random CE 10.5966) | logits std 0.639

Testing configs (per-GPU tokens/step = 16,384)...
  batch 8 ckpt_loss=False: OK, peak 9.50 GB
Using per-GPU batch 8 x accum 2 x 1 GPU(s) = 16,384 tok/step | loss ckpt: False | total 163,840,000 tokens

Compiling trunk, mode=default (a few minutes; the 'max_autotune_gemm' SM warning is harmless)...


W1006 00:46:29.041000 24 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


torch.compile: ENABLED

STARTING TRAINING
STEP     1/10000 | loss 10.8511 | lr 6.00e-07 | grad 19.64 | 11,786 tok/s | ETA 3.86 h
STEP    50/10000 | loss 7.8660 | lr 3.00e-05 | grad 3.32 | 12,138 tok/s | ETA 3.73 h
STEP   100/10000 | loss 6.5580 | lr 6.00e-05 | grad 2.71 | 11,246 tok/s | ETA 4.01 h
STEP   150/10000 | loss 5.6775 | lr 9.00e-05 | grad 2.25 | 11,016 tok/s | ETA 4.07 h
STEP   200/10000 | loss 6.0116 | lr 1.20e-04 | grad 5.73 | 10,912 tok/s | ETA 4.09 h
STEP   250/10000 | loss 4.8856 | lr 1.50e-04 | grad 1.60 | 10,847 tok/s | ETA 4.09 h
STEP   300/10000 | loss 5.3024 | lr 1.80e-04 | grad 2.12 | 10,804 tok/s | ETA 4.09 h
STEP   350/10000 | loss 4.6345 | lr 2.10e-04 | grad 1.32 | 10,767 tok/s | ETA 4.08 h
STEP   400/10000 | loss 4.4524 | lr 2.40e-04 | grad 1.63 | 10,749 tok/s | ETA 4.06 h
STEP   450/10000 | loss 4.4680 | lr 2.70e-04 | grad 1.54 | 10,726 tok/s | ETA 4.05 h
STEP   500/10000 | loss 4.0782 | lr 3.00e-04 | grad 1.32 | 10,714 tok/s | ETA 4.04 h

STEP 500 VAL | mean 